In [12]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
# Import the processing module from the same folder
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = True

In [13]:
def create_envelope(s_ed, s_uc, group_by = ['configuration', 'µ', 'iteration', 'day', 'hour,', 'r_id']):
    # Copy the relevant columns from s_ed['storage']
    envelope = s_ed['storage'][group_by + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()

    # Perform the first left join
    envelope = envelope.merge(
        s_uc['storage'][[col for col in group_by if col != 'iteration'] + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].rename(
            columns={'SOE_MWh': 'SOE_DA_MWh', 'envelope_up_MWh': 'envelope_up_DA_MWh', 'envelope_down_MWh': 'envelope_down_DA_MWh'}
        ),
        on=[col for col in group_by if col != 'iteration'],
        how='left'
    )
    
    # Perform the second left join
    envelope = envelope.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh', 'initial_energy_proportion']].drop_duplicates(),
        on='r_id',
        how='left'
    )
    
    # Calculate initial state of energy (SOE) based on maximum SOE and initial energy proportion
    envelope['SOE_0_MWh'] = envelope['SOE_max_MWh'] * envelope['initial_energy_proportion']
    envelope['SOC'] = envelope['SOE_MWh'] / envelope['SOE_max_MWh'] 
    # Group by day, configuration, and resource ID, and get the last entry for each group

    return envelope

In [14]:

ss = [
    # {'solution_folder': f"RTS-GMLC_v5.2.2s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2z", 'model_type' : 'envelope'}, # last opertional
    # {'solution_folder': f"RTS-GMLC_v5.2.2.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v6.2.2s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v15.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.0.1su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.0.2su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.1su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v16.2su", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v18.0.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v24.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_benchmark_v5.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_compare_v6.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v9.0su", 'VLGEN': 30, 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v25.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v19.0.3s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v6.2.3s", 'model_type' : 'e-reserve'}
    ]

s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    s_uc_name = 's_suc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_uc_ = load_solutions(s_uc_name, os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_uc.append(s_uc_)
    # s_ed.append(s_ed_)

    gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], solution_id = s) 

    gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], solution_id = s)

    gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

# s_uc = combine_solutions(s_uc)
# s_ed = combine_solutions(s_ed)
gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

if 'µ' in gcdi_KPI_adequacy.columns: 
#         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
    gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)
    gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)




../output/RTS-GMLC_v24.1s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v24.1s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v25.1s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v25.1s/all_gcdi_KPI_adequacy.parquet


In [15]:
gcdi_KPI_adequacy['model_type'].unique()

array(['envelope', 'conservative', 'e-reserve'], dtype=object)

In [16]:
if G_save:
    out= gcd_KPI_adequacy.copy()
    if 'µ' in out.columns: 
        out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
        # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: ((x[0] =='envelope')*(x[1]==1)*'conservatice' + x[0]), axis = 1)

    renames = {'µ': 'mu', 'ρ' : 'rho'}
    renames = {k: v for k, v in renames.items() if k in out.columns}
    out.rename(columns = renames, inplace = True)
    out.to_csv('gcd_KPI_adequacy.csv', index=False)
    gcdi_KPI_adequacy.rename(columns = renames, inplace = True)
    gcdi_KPI_adequacy.reset_index().to_csv('gcdi_KPI_adequacy.csv', index=False)

/tmp/ipykernel_12143/3797213802.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)


In [17]:
gcdi_KPI_adequacy

,model_type,solution_id,configuration,day,iteration,LLD_h,ENS_MWh,input_load_MWh,CURD_h,CUR_MWh,...,slack_energy_reserve_down_uc_MWh,required_energy_reserve_up_uc_MWh,slack_energy_reserve_up_uc_MWh,required_energy_reserve_down_uc_MWh,energy_reserve_down_uc_MWh,energy_reserve_up_uc_MWh,thermal_energy_reserve_down_uc_MWh,thermal_energy_reserve_up_uc_MWh,storage_energy_reserve_down_uc_MWh,storage_energy_reserve_up_uc_MWh
0,envelope,RTS-GMLC_v24.1s,base_ramp_storage_envelopes_up_0_54_dn_0_54,1,demand_1,0,0.000000,45565.932067,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,conservative,RTS-GMLC_v24.1s,base_ramp_storage_envelopes_up_1_dn_1,1,demand_1,0,0.000000,45565.932067,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,conservative,RTS-GMLC_v24.1s,base_ramp_storage_envelopes_up_1_dn_1,2,demand_1,0,0.000000,42264.410558,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,envelope,RTS-GMLC_v24.1s,base_ramp_storage_envelopes_up_0_49_dn_0_49,2,demand_1,0,0.000000,42264.410558,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,envelope,RTS-GMLC_v24.1s,base_ramp_storage_envelopes_up_0_63_dn_0_63,3,demand_1,0,0.000000,39266.476180,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,e-reserve,RTS-GMLC_v25.1s,base_ramp_storage_envelopes_up_1_dn_1,288,demand_1,9,4177.393863,75421.768308,0,0.0,...,0.0,7.617387e+05,0.0,1.255944e+06,1.255944e+06,7.617387e+05,352959.139157,10879.479037,9.029848e+05,7.508593e+05
288,e-reserve,RTS-GMLC_v25.1s,base_ramp_storage_envelopes_up_1_dn_1,289,demand_1,12,4666.733146,76165.466504,0,0.0,...,0.0,7.493366e+05,0.0,1.296587e+06,1.296587e+06,7.493366e+05,400985.163365,8789.053542,8.956017e+05,7.405476e+05
289,e-reserve,RTS-GMLC_v25.1s,base_ramp_storage_envelopes_up_1_dn_1,290,demand_1,17,8119.404291,79590.529870,0,0.0,...,0.0,7.938063e+05,0.0,1.364500e+06,1.364500e+06,7.938063e+05,387704.703927,8973.996236,9.767953e+05,7.848323e+05
290,e-reserve,RTS-GMLC_v25.1s,base_ramp_storage_envelopes_up_1_dn_1,291,demand_1,0,0.000000,59263.634622,0,0.0,...,0.0,7.797372e+05,0.0,1.468575e+06,1.468575e+06,7.797372e+05,421463.893026,3661.882702,1.047111e+06,7.760753e+05
